In [ ]:
import pandas as pd

df = pd.read_csv("../data/ufc-master.csv")

print("Rows and columns:", df.shape)
df.head()

Filas y columnas: (7177, 118)


,R_fighter,B_fighter,R_odds,B_odds,R_ev,B_ev,date,location,country,Winner,...,finish_details,finish_round,finish_round_time,total_fight_time_secs,r_dec_odds,b_dec_odds,r_sub_odds,b_sub_odds,r_ko_odds,b_ko_odds
0,Israel Adesanya,Joe Pyfer,-130.0,102.0,76.9231,102.0000,2026-03-28,"Seattle, Washington, USA",USA,Blue,...,Punches,2.0,4:18,558.0,163.0,900.0,2500.0,400.0,300.0,250.0
1,Alexa Grasso,Maycee Barber,124.0,-158.0,124.0000,63.2911,2026-03-28,"Seattle, Washington, USA",USA,Red,...,Punch,1.0,2:42,162.0,175.0,105.0,1400.0,800.0,2500.0,500.0
2,Michael Chiesa,Niko Price,-901.0,550.0,11.0988,550.0000,2026-03-28,"Seattle, Washington, USA",USA,Red,...,Rear Naked Choke,1.0,1:03,63.0,225.0,900.0,-150.0,1600.0,600.0,1000.0
3,Julian Erosa,Lerryan Douglas,235.0,-320.0,235.0000,31.2500,2026-03-28,"Seattle, Washington, USA",USA,Blue,...,Punches,1.0,3:33,213.0,600.0,500.0,600.0,2000.0,700.0,-150.0
4,Mansur Abdul-Malik,Yousri Belgaroui,-158.0,124.0,63.2911,124.0000,2026-03-28,"Seattle, Washington, USA",USA,Blue,...,Knee,3.0,3:39,819.0,350.0,240.0,800.0,1800.0,240.0,250.0


In [ ]:
print(df.columns.tolist())

['R_fighter', 'B_fighter', 'R_odds', 'B_odds', 'R_ev', 'B_ev', 'date', 'location', 'country', 'Winner', 'title_bout', 'weight_class', 'gender', 'no_of_rounds', 'B_current_lose_streak', 'B_current_win_streak', 'B_draw', 'B_avg_SIG_STR_landed', 'B_avg_SIG_STR_pct', 'B_avg_SUB_ATT', 'B_avg_TD_landed', 'B_avg_TD_pct', 'B_longest_win_streak', 'B_losses', 'B_total_rounds_fought', 'B_total_title_bouts', 'B_win_by_Decision_Majority', 'B_win_by_Decision_Split', 'B_win_by_Decision_Unanimous', 'B_win_by_KO/TKO', 'B_win_by_Submission', 'B_win_by_TKO_Doctor_Stoppage', 'B_wins', 'B_Stance', 'B_Height_cms', 'B_Reach_cms', 'B_Weight_lbs', 'R_current_lose_streak', 'R_current_win_streak', 'R_draw', 'R_avg_SIG_STR_landed', 'R_avg_SIG_STR_pct', 'R_avg_SUB_ATT', 'R_avg_TD_landed', 'R_avg_TD_pct', 'R_longest_win_streak', 'R_losses', 'R_total_rounds_fought', 'R_total_title_bouts', 'R_win_by_Decision_Majority', 'R_win_by_Decision_Split', 'R_win_by_Decision_Unanimous', 'R_win_by_KO/TKO', 'R_win_by_Submission',

In [ ]:
df.isnull().sum().sort_values(ascending=False).head(20)

B_Women's Featherweight_rank    7176
R_Women's Featherweight_rank    7168
B_Pound-for-Pound_rank          7096
B_Women's Flyweight_rank        7092
R_Women's Flyweight_rank        7066
B_Women's Strawweight_rank      7063
B_Women's Bantamweight_rank     7057
B_Welterweight_rank             7046
B_Light Heavyweight_rank        7044
B_Featherweight_rank            7043
B_Lightweight_rank              7042
B_Bantamweight_rank             7042
B_Flyweight_rank                7034
B_Middleweight_rank             7026
B_Heavyweight_rank              7018
R_Women's Strawweight_rank      7013
R_Women's Bantamweight_rank     7009
R_Featherweight_rank            6979
R_Lightweight_rank              6975
R_Middleweight_rank             6974
dtype: int64

In [ ]:
import numpy as np
import plotly.express as px

# Keep only fights with a winner (remove draws and no contests)
df = df[df["Winner"].isin(["Red", "Blue"])].copy()
print("Fights with a winner:", len(df))

Fights with a winner: 7169


QUESTION 1,How often does the betting favourite win?

In [ ]:

odds = df.dropna(subset=["R_odds", "B_odds"])        # fights with betting odds
odds = odds[odds["R_odds"] != odds["B_odds"]]        # remove fights without a clear favourite

favourite = np.where(odds["R_odds"] < odds["B_odds"], "Red", "Blue")  # lower odds = favourite
favourite_win_rate = (favourite == odds["Winner"]).mean()

print("Fights analysed:", len(odds))
print(f"The favourite wins {favourite_win_rate:.1%} of the time")

Fights analysed: 6796
The favourite wins 66.6% of the time


**Q1 – Conclusion:** The betting favourite wins 66.6% of fights (6,796 fights analysed). Bookmakers are quite accurate, but 1 in 3 fights is an upset. This 66.6% is the benchmark our prediction model has to beat.

QUESTION 2: Does reach help?

In [ ]:
reach = df.dropna(subset=["R_Reach_cms", "B_Reach_cms"]).copy()
reach = reach[reach["R_Reach_cms"] != reach["B_Reach_cms"]]   # remove fights with equal reach

reach["reach_diff"] = (reach["R_Reach_cms"] - reach["B_Reach_cms"]).abs()
longer_reach = np.where(reach["R_Reach_cms"] > reach["B_Reach_cms"], "Red", "Blue")
reach["longer_reach_won"] = longer_reach == reach["Winner"]

# Group fights by size of the reach difference
reach["diff_group"] = pd.cut(reach["reach_diff"], [0, 5, 10, 15, 100],
                             labels=["0-5 cm", "5-10 cm", "10-15 cm", "15+ cm"])
reach_table = (reach.groupby("diff_group", observed=True)["longer_reach_won"]
               .agg(win_rate="mean", fights="count").reset_index())
reach_table["win_rate"] = reach_table["win_rate"] * 100
print(reach_table)

px.bar(reach_table, x="diff_group", y="win_rate", text_auto=".1f",
       labels={"diff_group": "Reach difference", "win_rate": "Win rate of longer-reach fighter (%)"},
       title="Does a reach advantage help to win?")

  diff_group   win_rate  fights
0     0-5 cm  49.103139    1784
1    5-10 cm  51.449843    2552
2   10-15 cm  53.162530    1249
3     15+ cm  57.864358     693


**Q2 – Conclusion:** A reach advantage only matters when it is large. With a difference of 0–5 cm the longer-reach fighter wins 49% (no advantage), but with 15+ cm the win rate rises to 57.9%. The effect grows steadily with the size of the difference.

QUESTION 3: How do the fights in each weight class end?

In [ ]:

finish_table = pd.crosstab(df["weight_class"], df["finish"], normalize="index") * 100
finish_table = finish_table[["KO/TKO", "SUB", "U-DEC", "S-DEC"]].sort_values("KO/TKO")

px.bar(finish_table, orientation="h", barmode="stack",
       labels={"value": "% of fights", "weight_class": "Weight class", "finish": "Finish method"},
       title="How do fights end in each weight class?")

**Q3 – Conclusion:** The heavier the weight class, the more knockouts. Over 50% of heavyweight fights end by KO/TKO, compared with less than 15% in women's strawweight. Women's fights end by decision much more often, while the submission rate is similar across all divisions.

QUESTION 4: In which weight classes are upsets most common?


In [ ]:
odds["favourite"] = np.where(odds["R_odds"] < odds["B_odds"], "Red", "Blue")
odds["upset"] = odds["favourite"] != odds["Winner"]

upsets = (odds.groupby("weight_class")["upset"]
          .agg(upset_rate="mean", fights="count").reset_index())
upsets = upsets[upsets["fights"] >= 100]            # only classes with enough fights
upsets["upset_rate"] = upsets["upset_rate"] * 100
upsets = upsets.sort_values("upset_rate")

px.bar(upsets, x="upset_rate", y="weight_class", orientation="h",
       text_auto=".1f", hover_data=["fights"],
       labels={"upset_rate": "Upset rate (%)", "weight_class": "Weight class", "fights": "Fights"},
       title="In which weight classes are upsets most common?")

**Q4 – Conclusion:** Upset rates are similar across weight classes (30.7%–36.9%), so the weight class has little influence on how predictable a fight is. Featherweight and light heavyweight have the most upsets; bantamweight and flyweight the fewest. Only classes with 100+ fights were included.

QUESTION 5: If you had bet €10 on every fight since 2010, would you make money?


In [ ]:
bets = odds.dropna(subset=["R_ev", "B_ev"]).sort_values("date").copy()

# R_ev / B_ev = profit of a winning 100 bet -> divide by 10 for a 10 bet
fav_profit_if_win = np.where(bets["favourite"] == "Red", bets["R_ev"], bets["B_ev"]) / 10
dog_profit_if_win = np.where(bets["favourite"] == "Red", bets["B_ev"], bets["R_ev"]) / 10

bets["Always bet on the favourite"] = np.where(~bets["upset"], fav_profit_if_win, -10).cumsum()
bets["Always bet on the underdog"] = np.where(bets["upset"], dog_profit_if_win, -10).cumsum()

total_bet = len(bets) * 10
for strategy in ["Always bet on the favourite", "Always bet on the underdog"]:
    final = bets[strategy].iloc[-1]
    print(f"{strategy}: €{final:,.0f} after betting €{total_bet:,} ({final / total_bet:.1%} return)")

px.line(bets, x="date", y=["Always bet on the favourite", "Always bet on the underdog"],
        labels={"value": "Accumulated profit (€)", "date": "Date", "variable": "Strategy"},
        title="€10 on every UFC fight since 2010: would you make money?")

Always bet on the favourite: €-1,961 after betting €67,960 (-2.9% return)
Always bet on the underdog: €-5,139 after betting €67,960 (-7.6% return)


**Q5 – Conclusion:** Betting €10 on every fight since 2010 loses money with both simple strategies: about −€1,960 (−2.9%) always betting on the favourite and about −€5,140 (−7.6%) always betting on the underdog. This reflects the bookmaker's margin: without an edge, betting loses in the long run. Betting on underdogs was roughly break-even until around 2017 and then declined sharply, possibly because bookmakers' odds became more accurate (hypothesis). To make money, a strategy needs information the odds do not fully reflect.

QUESTION 6a: Does the younger fighter win more often?


In [ ]:

age = df.dropna(subset=["R_age", "B_age"]).copy()
age = age[age["R_age"] != age["B_age"]]
age["age_gap"] = (age["R_age"] - age["B_age"]).abs()
age["younger_won"] = np.where(age["R_age"] < age["B_age"], "Red", "Blue") == age["Winner"]
age["gap_group"] = pd.cut(age["age_gap"], [0, 2, 5, 8, 30],
                          labels=["1-2 years", "3-5 years", "6-8 years", "9+ years"])

age_table = (age.groupby("gap_group", observed=True)["younger_won"]
             .agg(win_rate="mean", fights="count").reset_index())
age_table["win_rate"] = age_table["win_rate"] * 100

px.bar(age_table, x="gap_group", y="win_rate", text_auto=".1f", hover_data=["fights"],
       labels={"gap_group": "Age gap", "win_rate": "Win rate of younger fighter (%)", "fights": "Fights"},
       title="Does the younger fighter win more often?")

QUESTION 6b: Win rate by age

In [ ]:

fighters = pd.concat([
    pd.DataFrame({"age": df["R_age"], "won": df["Winner"] == "Red"}),
    pd.DataFrame({"age": df["B_age"], "won": df["Winner"] == "Blue"}),
]).dropna()
fighters["age_group"] = pd.cut(fighters["age"], [17, 24, 27, 30, 33, 36, 50],
                               labels=["<25", "25-27", "28-30", "31-33", "34-36", "37+"])

age_curve = (fighters.groupby("age_group", observed=True)["won"]
             .agg(win_rate="mean", fights="count").reset_index())
age_curve["win_rate"] = age_curve["win_rate"] * 100

px.line(age_curve, x="age_group", y="win_rate", markers=True, hover_data=["fights"],
        labels={"age_group": "Age at fight", "win_rate": "Win rate (%)", "fights": "Fights"},
        title="At what age do UFC fighters win the most?")

**Q6 – Conclusion:** Age is one of the strongest factors found so far. With an age gap of 9+ years, the younger fighter wins 67.3% of fights, as often as the betting favourite. The win rate falls steadily with age, from 58% under 25 to 38% at 37+. Note: this is partly a selection effect, since young UFC fighters are often top prospects and many older fighters are at the end of their careers.

QUESTION 7: When do knockouts happen? 3-round vs 5-round fights

In [ ]:

ko = df[df["finish"] == "KO/TKO"].dropna(subset=["finish_round", "finish_round_time"]).copy()
ko = ko[ko["no_of_rounds"].isin([3, 5])]                   # remove old 4-round fights
ko = ko[ko["finish_round"] <= ko["no_of_rounds"]]          # remove data errors

# Convert "4:18" into seconds, then into the minute of the round (5:00 -> minute 5)
time_parts = ko["finish_round_time"].str.split(":")
ko["seconds"] = time_parts.str[0].astype(int) * 60 + time_parts.str[1].astype(int)
ko["minute"] = np.ceil(ko["seconds"] / 60).clip(lower=1).astype(int)
ko["round"] = "Round " + ko["finish_round"].astype(int).astype(str)
ko["fight_type"] = np.where(ko["no_of_rounds"] == 5,
                            "5-round fights (main events & title fights)", "3-round fights")

for fight_type, group in ko.groupby("fight_type"):
    first_round_share = (group["finish_round"] == 1).mean()
    print(f"{fight_type}: {len(group)} KOs, {first_round_share:.1%} of them in round 1")

    table = pd.crosstab(group["round"], group["minute"], normalize="all") * 100
    px.imshow(table.round(1), text_auto=True, color_continuous_scale="Reds", aspect="auto",
              labels={"x": "Minute of the round", "y": "Round", "color": "% of KOs"},
              title=f"When do knockouts happen? – {fight_type}").show()

3-round fights: 1823 KOs, 53.5% of them in round 1


5-round fights (main events & title fights): 262 KOs, 42.4% of them in round 1


**Q7– Conclusion:** Knockouts are front-loaded: in 3-round fights, 53.5% of KOs happen in round 1, and round 3 is almost three times less dangerous. There is a clear spike in the last minute of rounds 1 and 2 (11.8% and 9.5%), which reflects late-round urgency and stoppages between rounds (53 KOs are recorded at exactly 5:00). In 5-round fights (main events and title fights), only 42.4% of KOs happen in round 1 and they peak later (minute 3), suggesting that elite fighters pace themselves more. Insight for the simulator: KO risk is not constant – it depends on the round, the minute and the fight length.

QUESTION 8: Did fights without fans (COVID, 2020-2021) change anything?


In [ ]:

covid = df[df["empty_arena"].isin([0, 1])].copy()
covid["crowd"] = np.where(covid["empty_arena"] == 1, "No fans (COVID)", "With fans")

# a) Did the favourite win less often?
c_odds = covid.dropna(subset=["R_odds", "B_odds"])
c_odds = c_odds[c_odds["R_odds"] != c_odds["B_odds"]].copy()
c_odds["favourite_won"] = np.where(c_odds["R_odds"] < c_odds["B_odds"], "Red", "Blue") == c_odds["Winner"]
print((c_odds.groupby("crowd")["favourite_won"].mean() * 100).round(1))

# b) Did fights end differently?
crowd_table = (pd.crosstab(covid["crowd"], covid["finish"], normalize="index") * 100)[["KO/TKO", "SUB", "U-DEC", "S-DEC"]]
crowd_long = crowd_table.reset_index().melt(id_vars="crowd", var_name="finish", value_name="pct")

px.bar(crowd_long, x="finish", y="pct", color="crowd", barmode="group", text_auto=".1f",
       labels={"finish": "Finish method", "pct": "% of fights", "crowd": ""},
       title="Did fights without fans end differently?")

crowd
No fans (COVID)    64.8
With fans          65.6
Name: favourite_won, dtype: float64


**Q8:(discarded) – Conclusion:** Fights without fans (COVID, 2020–2021) were almost identical to fights with fans: the favourite won 64.8% vs 65.6%, and finish methods were very similar. The only notable difference is fewer split decisions (8.1% vs 10.7%), which could suggest that crowds influence judges in close fights. However, most no-fan events took place at the UFC Apex, which has a smaller octagon, so the effect of the crowd cannot be separated from the effect of the cage size. **Decision:** this question was replaced by "Do fighters recover after being knocked out?", which gives stronger and more useful insights for the project.

# Build a fighter history: one row per fighter per fight, in date order


In [ ]:
rows = []
for corner, colour, opponent in [("R", "Red", "B"), ("B", "Blue", "R")]:
    rows.append(pd.DataFrame({
        "fighter": df[f"{corner}_fighter"],
        "date": pd.to_datetime(df["date"]),
        "age": df[f"{corner}_age"],
        "won": df["Winner"] == colour,
        "finish": df["finish"],
        "odds": df[f"{corner}_odds"],
        "opp_odds": df[f"{opponent}_odds"],
    }))

fighters_hist = pd.concat(rows).sort_values(["fighter", "date"]).reset_index(drop=True)
fighters_hist["lost_by_ko"] = (~fighters_hist["won"]) & (fighters_hist["finish"] == "KO/TKO")

# Look at each fighter's PREVIOUS fight
previous = fighters_hist.groupby("fighter")
fighters_hist["prev_won"] = previous["won"].shift(1)
fighters_hist["prev_lost_by_ko"] = previous["lost_by_ko"].shift(1)
fighters_hist["days_since_last_fight"] = (fighters_hist["date"] - previous["date"].shift(1)).dt.days

print("Rows (fighter + fight):", len(fighters_hist))
fighters_hist.head(10)

Rows (fighter + fight): 14338


,fighter,date,age,won,finish,odds,opp_odds,lost_by_ko,prev_won,prev_lost_by_ko,days_since_last_fight
0,Jun Yong Park,2019-12-21,28,True,NaN,-138.0,105.0,False,NaN,NaN,NaN
1,AJ Cunningham,2024-03-02,29,False,KO/TKO,800.0,140.0,True,NaN,NaN,NaN
2,AJ Cunningham,2025-03-15,30,False,U-DEC,410.0,-550.0,False,False,True,378.0
3,AJ Dobson,2022-02-12,30,False,U-DEC,-110.0,-110.0,False,NaN,NaN,NaN
4,AJ Dobson,2022-10-22,30,False,U-DEC,185.0,-215.0,False,False,False,252.0
5,AJ Dobson,2023-08-12,31,True,U-DEC,124.0,-148.0,False,False,False,294.0
6,AJ Dobson,2024-03-23,32,False,KO/TKO,160.0,-192.0,True,True,False,224.0
7,AJ Fletcher,2022-03-12,25,False,U-DEC,165.0,-195.0,False,NaN,NaN,NaN
8,AJ Fletcher,2022-08-20,25,False,U-DEC,-165.0,140.0,False,False,False,161.0
9,AJ Fletcher,2023-02-18,26,True,SUB,-350.0,290.0,False,False,False,182.0


 Q9:Do fighters recover after being knocked out?


In [ ]:
ko_rec = fighters_hist.dropna(subset=["prev_won"]).copy()
ko_rec["previous_fight"] = np.select(
    [ko_rec["prev_lost_by_ko"] == True, ko_rec["prev_won"] == False],
    ["Lost by KO", "Lost (other way)"], default="Won")

# What the bookmakers expected: implied win probability from American odds
def implied_prob(odds):
    return np.where(odds < 0, -odds / (-odds + 100), 100 / (odds + 100))

p_fighter = implied_prob(ko_rec["odds"])
p_opponent = implied_prob(ko_rec["opp_odds"])
ko_rec["expected_win"] = p_fighter / (p_fighter + p_opponent)   # remove the bookmaker margin

summary = (ko_rec.groupby("previous_fight")
           .agg(actual_win_rate=("won", "mean"),
                expected_by_bookmakers=("expected_win", "mean"),
                knocked_out_again=("lost_by_ko", "mean"),
                fights=("won", "count"))
           .reindex(["Won", "Lost (other way)", "Lost by KO"]))
summary[["actual_win_rate", "expected_by_bookmakers", "knocked_out_again"]] *= 100
print(summary.round(1))

# Chart 1: actual win rate vs what the bookmakers expected
chart = summary.reset_index().melt(id_vars=["previous_fight", "fights"],
                                   value_vars=["actual_win_rate", "expected_by_bookmakers"],
                                   var_name="measure", value_name="pct")
px.bar(chart, x="previous_fight", y="pct", color="measure", barmode="group",
       text_auto=".1f", hover_data=["fights"],
       labels={"previous_fight": "Result of previous fight", "pct": "Win rate (%)", "measure": ""},
       title="Do fighters recover after being knocked out?").show()

# Chart 2: chance of being knocked out again
px.bar(summary.reset_index(), x="previous_fight", y="knocked_out_again",
       text_auto=".1f", hover_data=["fights"],
       labels={"previous_fight": "Result of previous fight", "knocked_out_again": "Lost by KO in next fight (%)"},
       title="Once knocked out, more likely to be knocked out again?").show()

                  actual_win_rate  expected_by_bookmakers  knocked_out_again  \
previous_fight                                                                 
Won                          52.5                    52.1               14.7   
Lost (other way)             49.6                    49.2               13.9   
Lost by KO                   45.0                    47.0               22.4   

                  fights  
previous_fight            
Won                 6735  
Lost (other way)    3727  
Lost by KO          1635  


**Q9– Conclusion:** Being knocked out leaves a mark. Fighters coming off a KO loss win only 45.0% of their next fight (vs 52.5% after a win), and 22.4% are knocked out again (vs 14.7%) – about 50% more. Losing by decision or submission does not increase this risk (13.9%), so the effect is specific to KOs. Bookmakers price form very accurately for winners (52.1% expected vs 52.5% actual) and other losers (49.2% vs 49.6%), but overrate fighters coming off a KO (47.0% expected vs 45.0% actual). With 1,635 cases this is a hint, not proof – it will be tested as a betting strategy. Note: we cannot fully separate accumulated damage from some fighters simply having a weaker chin.

**Link Q6 + Q9: Do older fighters get knocked out more often?**


In [ ]:
age_ko = fighters_hist.dropna(subset=["age"]).copy()
age_ko["age_group"] = pd.cut(age_ko["age"], [17, 27, 30, 33, 36, 50],
                             labels=["<28", "28-30", "31-33", "34-36", "37+"])
chin = (age_ko.groupby("age_group", observed=True)["lost_by_ko"]
        .agg(ko_loss_rate="mean", fights="count").reset_index())
chin["ko_loss_rate"] *= 100

px.line(chin, x="age_group", y="ko_loss_rate", markers=True, hover_data=["fights"],
        labels={"age_group": "Age at fight", "ko_loss_rate": "Lost by KO (%)", "fights": "Fights"},
        title="Does the chin fade with age? (% of fights lost by KO)")

**Q6 + Q9 – Does the chin fade with age?** The share of fights lost by KO doubles with age, from 10.9% under 28 to 23.3% at 37+. Older fighters do not only win less often – they also lose in a harsher way. Hypothesis for the next step: veterans coming off a KO loss may be in a "danger zone".

 THE MONEY TEST: can our findings beat the bookmakers? (€10 per bet)


In [ ]:

def bet_profit(bet_odds, bet_won, stake=10):
    """Profit of a bet with American odds: the payout if won, minus the stake if lost."""
    bet_odds = np.asarray(bet_odds, dtype=float)
    payout = np.where(bet_odds > 0, stake * bet_odds / 100, stake * 100 / -bet_odds)
    return np.where(bet_won, payout, -stake)

bets_log = []   # every single bet of every strategy

def add_strategy(name, dates, bet_odds, bet_won):
    bets_log.append(pd.DataFrame({
        "strategy": name,
        "date": pd.to_datetime(dates),
        "won": np.asarray(bet_won, dtype=bool),
        "profit": bet_profit(bet_odds, bet_won),
    }))

# Baselines (Q5)
fav_odds = np.where(odds["favourite"] == "Red", odds["R_odds"], odds["B_odds"])
dog_odds = np.where(odds["favourite"] == "Red", odds["B_odds"], odds["R_odds"])
add_strategy("Baseline: always the favourite", odds["date"], fav_odds, odds["favourite"] == odds["Winner"])
add_strategy("Baseline: always the underdog", odds["date"], dog_odds, odds["favourite"] != odds["Winner"])

# Finding 1 (Q6): bet on the younger fighter when the age gap is 5+ years
s = odds.dropna(subset=["R_age", "B_age"])
s = s[(s["R_age"] - s["B_age"]).abs() >= 5]
younger = np.where(s["R_age"] < s["B_age"], "Red", "Blue")
add_strategy("Age: younger fighter (5+ years gap)", s["date"],
             np.where(younger == "Red", s["R_odds"], s["B_odds"]), younger == s["Winner"])

# Finding 2 (Q2): bet on the longer reach when the gap is 10+ cm
s = odds.dropna(subset=["R_Reach_cms", "B_Reach_cms"])
s = s[(s["R_Reach_cms"] - s["B_Reach_cms"]).abs() >= 10]
longer = np.where(s["R_Reach_cms"] > s["B_Reach_cms"], "Red", "Blue")
add_strategy("Reach: longer reach (10+ cm gap)", s["date"],
             np.where(longer == "Red", s["R_odds"], s["B_odds"]), longer == s["Winner"])

# Finding 3 (Q9): bet AGAINST a fighter coming off a KO loss
s = fighters_hist[fighters_hist["prev_lost_by_ko"] == True].dropna(subset=["odds", "opp_odds"])
add_strategy("KO: against a fighter coming off a KO loss", s["date"], s["opp_odds"], ~s["won"])

# Results per strategy
bets_log = pd.concat(bets_log)
results = (bets_log.groupby("strategy")
           .agg(bets=("profit", "size"), win_rate=("won", "mean"), profit=("profit", "sum"))
           .reset_index())
results["win_rate"] *= 100
results["roi"] = results["profit"] / (results["bets"] * 10) * 100
results = results.sort_values("roi")
print(results.round(1).to_string(index=False))

results["result"] = np.where(results["roi"] > 0, "Makes money", "Loses money")
px.bar(results, x="roi", y="strategy", orientation="h", color="result", text_auto=".1f",
       hover_data=["bets", "win_rate", "profit"],
       color_discrete_map={"Makes money": "green", "Loses money": "red"},
       labels={"roi": "Return on investment (%)", "strategy": "", "result": ""},
       title="Can our findings beat the bookmakers? (€10 per bet)")

                                  strategy  bets  win_rate  profit  roi
             Baseline: always the underdog  6796      33.4 -5135.3 -7.6
            Baseline: always the favourite  6796      66.6 -1960.0 -2.9
          Reach: longer reach (10+ cm gap)  1847      54.9  -382.2 -2.1
KO: against a fighter coming off a KO loss  1567      54.9    -3.2 -0.0
       Age: younger fighter (5+ years gap)  2672      61.8   408.3  1.5


 How did each strategy's money evolve over time?


In [ ]:
timeline = bets_log.sort_values("date").copy()
timeline["accumulated_profit"] = timeline.groupby("strategy")["profit"].cumsum()

px.line(timeline, x="date", y="accumulated_profit", color="strategy",
        labels={"date": "Date", "accumulated_profit": "Accumulated profit (€)", "strategy": "Strategy"},
        title="The money test over time: €10 per bet since 2010")

**The money test – Conclusion:** Using the bookmakers' odds as a benchmark, only one of our findings beats the market: betting on the younger fighter when the age gap is 5+ years gives a +1.5% return (2,672 bets, 61.8% win rate). Betting against fighters coming off a KO loss breaks even (0.0%), which already beats the bookmaker's margin, while the reach strategy loses (−2.1%), meaning reach is already priced into the odds. Key lesson: winning often is not the same as making money – the favourite wins 66.6% of the time but loses money, because what matters is winning more often than the odds imply. Over time, the age and KO edges peaked around 2018–2020 and have been shrinking since, like the underdog strategy in Q3: the market seems to learn. With ~2,700 bets, a ±2% return can happen by chance, so this is a strong hint, not proof – it will be tested across time periods.

# MONEY TEST – STEP 2: Is the edge real, or just luck? Check it in different periods


In [ ]:
bets_log["period"] = pd.cut(bets_log["date"].dt.year, [2009, 2015, 2020, 2026],
                            labels=["2010-2015", "2016-2020", "2021-2026"])

by_period = (bets_log.groupby(["strategy", "period"], observed=True)
             .agg(bets=("profit", "size"), profit=("profit", "sum"))
             .reset_index())
by_period["roi"] = by_period["profit"] / (by_period["bets"] * 10) * 100
print(by_period.pivot(index="strategy", columns="period", values="roi").round(1))

px.bar(by_period, x="period", y="roi", color="strategy", barmode="group",
       text_auto=".1f", hover_data=["bets", "profit"],
       labels={"period": "Period", "roi": "Return on investment (%)", "strategy": "Strategy"},
       title="Money test by period: is the edge consistent over time?")

period                                      2010-2015  2016-2020  2021-2026
strategy                                                                   
Age: younger fighter (5+ years gap)               2.3        1.1        1.3
Baseline: always the favourite                   -4.0       -3.9       -0.9
Baseline: always the underdog                    -2.1       -6.7      -13.3
KO: against a fighter coming off a KO loss        3.0        3.7       -5.6
Reach: longer reach (10+ cm gap)                 -1.7       -1.0       -3.4


**Money test, step 2 – Conclusion:** Splitting the bets into three periods shows which edges are real. The age strategy is the only one that makes money in all three periods (+2.3%, +1.1%, +1.3%), so bookmakers seem to consistently underrate youth. The KO strategy made money for ten years (+3.0%, +3.7%) but lost 5.6% in 2021–2026: the market learned and corrected this bias. Reach loses in every period, so it is already priced in. A new finding appears in the baselines: betting on the favourite is now almost break-even (−0.9%), while betting on underdogs loses more and more (−13.3% in 2021–2026). This is the well-known favourite-longshot bias: the public overbets on big upsets, so bookmakers pay them less than they are worth. Implications for the prediction model: it must be tested on recent fights (2021–2026), and age should be one of its key features.